[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ucmoprj/RL_starcraft/blob/main/02_q_learning/sc2_colab.ipynb)

# Lesson 2 in StarCraft II, on Google Colab

This notebook trains the Lesson 2 Q-learning marine on **MoveToBeacon** and lets you watch it,
without installing anything on your own computer.

**How it differs from running on your PC**

- Colab is a Linux server with no screen, so we download the Linux build of StarCraft II
  (about 4 GB, a few minutes) each time the notebook starts.
- There is no game window. To *watch*, we record the game's screen data at every step and
  draw it as an animation: the marine, the beacon, and the direction the agent chose.
- The training code is exactly the same as `02_q_learning/q_learning.py`.

Run the cells from top to bottom with **Shift + Enter**. Setup takes about 5 minutes and
training about 10.

By downloading StarCraft II you accept the
[AI and Machine Learning License](https://blzdistsc2-a.akamaihd.net/AI_AND_MACHINE_LEARNING_LICENSE.html).

## 1. Get the course code and PySC2

In [ ]:
!git clone -q https://github.com/ucmoprj/RL_starcraft.git
%cd RL_starcraft
!pip install -q "pysc2 @ https://github.com/google-deepmind/pysc2/archive/refs/heads/master.zip"

## 2. Download StarCraft II (Linux) and the mini-game maps

In [ ]:
import os

SC2 = "/content/StarCraftII"
if not os.path.exists(SC2):
    !wget -q --show-progress http://blzdistsc2-a.akamaihd.net/Linux/SC2.4.10.zip -O /content/SC2.zip
    !unzip -q -P iagreetotheeula /content/SC2.zip -d /content/
    !rm /content/SC2.zip
    !wget -q https://github.com/deepmind/pysc2/releases/download/v1.2/mini_games.zip -O /content/mini_games.zip
    !mkdir -p {SC2}/Maps
    !unzip -q -o /content/mini_games.zip -d {SC2}/Maps/
    !rm /content/mini_games.zip

os.environ["SC2PATH"] = SC2
!ls {SC2}/Maps/mini_games

You should see `MoveToBeacon.SC2Map` in the list.

## 3. Train

The same command as on your PC. The average reward per 10 episodes should climb from
about 1 (random) to about 20. Use `--episodes 100` for a quicker, rougher run.

In [ ]:
!python 02_q_learning/q_learning.py --env sc2 --episodes 300

The learning curve and the learned policy map (compare them with Lesson 2, section 8):

In [ ]:
from IPython.display import Image, display
display(Image("02_q_learning/results/sc2_learning_curve.png", width=600))
display(Image("02_q_learning/results/sc2_policy.png", width=600))

## 4. Watch the trained marine

This plays one full episode (120 game seconds) with the learned Q-table, always choosing
the highest value (like `watch.py`). It records the game's `player_relative` screen layer at
each step, then draws it as an animation. Blue is the marine, orange is the beacon, and the
arrow is the direction the agent picked from its Q-table.

Set `RANDOM = True` to compare with a marine that moves at random.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import ListedColormap

sys.path.insert(0, ".")
from envs.move_to_beacon import DIRECTIONS, MoveToBeaconEnv

RANDOM = False

Q = np.load("02_q_learning/results/sc2_q_table.npy")
rng = np.random.default_rng(0)
env = MoveToBeaconEnv()
frames, total = [], 0.0
try:
    state, done = env.reset(), False
    while not done:
        action = int(rng.integers(env.n_actions)) if RANDOM else int(np.argmax(Q[state]))
        screen = np.array(env._timestep.observation.feature_screen.player_relative)
        frames.append((screen, action, total))
        state, reward, done = env.step(action)
        total += reward
finally:
    env.close()
print(f"total reward: {total:.0f} beacons in {len(frames)} steps")

# 0 = empty, 1 = our marine, 3 = the beacon (neutral)
cmap = ListedColormap(["#F4F6F8", "#20446E", "#20446E", "#E0A020", "#B03B74"])
fig, ax = plt.subplots(figsize=(4.5, 4.5))
img = ax.imshow(frames[0][0], cmap=cmap, vmin=0, vmax=4)
arrow = ax.annotate("", xy=(0, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color="#1C7C8F", lw=2))
dot = ax.scatter([], [], s=140, color="#20446E", zorder=3)
title = ax.set_title("")
ax.set_xticks([]); ax.set_yticks([])


def draw(i):
    screen, action, score = frames[i]
    img.set_data(screen)
    ys, xs = np.nonzero(screen == 1)
    if len(xs):
        x, y = xs.mean(), ys.mean()
        dx, dy = DIRECTIONS[action]
        dot.set_offsets([[x, y]])
        arrow.set_position((x, y))
        arrow.xy = (x + dx * 8, y + dy * 8)
    title.set_text(f"step {i + 1}   beacons {score:.0f}")
    return img, dot, arrow, title


anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=120)
plt.close(fig)

from IPython.display import HTML
HTML(anim.to_jshtml())

## Things to try

- Train again with `--alpha 0.5` or `--gamma 0.5` (section 3), then watch again (section 4).
- Set `RANDOM = True` in section 4 and compare the score.
- Each training run overwrites `sc2_q_table.npy`. Files on Colab disappear when the session
  ends; use the folder icon on the left to download anything you want to keep.